In [25]:
import snowflake.connector
import pandas as pd
import os

conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    schema=os.getenv("SNOWFLAKE_SCHEMA")
)

df = pd.read_sql("SELECT * FROM DIABETES_TABLE", conn)
df

/tmp/ipykernel_99/1136124994.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM DIABETES_TABLE", conn)


,ID,NO_PATION,GENDER,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI,CLASE
0,502,17975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221,M,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223,M,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,200,454317,M,71,11.0,97,7.0,7.5,1.7,1.2,1.8,0.6,30.0,Y
996,671,876534,M,31,3.0,60,12.3,4.1,2.2,0.7,2.4,15.4,37.2,Y
997,669,87654,M,30,7.1,81,6.7,4.1,1.1,1.2,2.4,8.1,27.4,Y
998,99,24004,M,38,5.8,59,6.7,5.3,2.0,1.6,2.9,14.0,40.5,Y


In [57]:
import matplotlib.pyplot as plt
import numpy as np
!pip install imbalanced-learn  # roda só se ainda não tiver instalado

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

In [27]:
diabetes = df.copy()
diabetes

,ID,NO_PATION,GENDER,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI,CLASE
0,502,17975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221,M,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656,F,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223,M,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,200,454317,M,71,11.0,97,7.0,7.5,1.7,1.2,1.8,0.6,30.0,Y
996,671,876534,M,31,3.0,60,12.3,4.1,2.2,0.7,2.4,15.4,37.2,Y
997,669,87654,M,30,7.1,81,6.7,4.1,1.1,1.2,2.4,8.1,27.4,Y
998,99,24004,M,38,5.8,59,6.7,5.3,2.0,1.6,2.9,14.0,40.5,Y


EDA

In [28]:
diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         1000 non-null   int64  
 1   NO_PATION  1000 non-null   int64  
 2   GENDER     1000 non-null   object 
 3   AGE        1000 non-null   int64  
 4   UREA       1000 non-null   float64
 5   CR         1000 non-null   int64  
 6   HBA1C      1000 non-null   float64
 7   CHOL       1000 non-null   float64
 8   TG         1000 non-null   float64
 9   HDL        1000 non-null   float64
 10  LDL        1000 non-null   float64
 11  VLDL       1000 non-null   float64
 12  BMI        1000 non-null   float64
 13  CLASE      1000 non-null   object 
dtypes: float64(8), int64(4), object(2)
memory usage: 109.5+ KB


In [29]:
diabetes.describe()

,ID,NO_PATION,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI
count,1000.000000,1.000000e+03,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,340.500000,2.705514e+05,53.528000,5.124743,68.943000,8.281160,4.862820,2.349610,1.204750,2.609790,1.854700,29.578020
std,240.397673,3.380758e+06,8.799241,2.935165,59.984747,2.534003,1.301738,1.401176,0.660414,1.115102,3.663599,4.962388
min,1.000000,1.230000e+02,20.000000,0.500000,6.000000,0.900000,0.000000,0.300000,0.200000,0.300000,0.100000,19.000000
25%,125.750000,2.406375e+04,51.000000,3.700000,48.000000,6.500000,4.000000,1.500000,0.900000,1.800000,0.700000,26.000000
50%,300.500000,3.439550e+04,55.000000,4.600000,60.000000,8.000000,4.800000,2.000000,1.100000,2.500000,0.900000,30.000000
75%,550.250000,4.538425e+04,59.000000,5.700000,73.000000,10.200000,5.600000,2.900000,1.300000,3.300000,1.500000,33.000000
max,800.000000,7.543566e+07,79.000000,38.900000,800.000000,16.000000,10.300000,13.800000,9.900000,9.900000,35.000000,47.750000


In [30]:
diabetes.isnull().sum()

ID           0
NO_PATION    0
GENDER       0
AGE          0
UREA         0
CR           0
HBA1C        0
CHOL         0
TG           0
HDL          0
LDL          0
VLDL         0
BMI          0
CLASE        0
dtype: int64

In [31]:
encoder = LabelEncoder()
diabetes['GENDER'] = encoder.fit_transform(diabetes['GENDER'])
diabetes.head()

,ID,NO_PATION,GENDER,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI,CLASE
0,502,17975,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,735,34221,1,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,420,47975,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,680,87656,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,504,34223,1,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N


In [32]:
diabetes.drop(['NO_PATION', 'ID'], axis=1, inplace=True)
diabetes.head()

,GENDER,AGE,UREA,CR,HBA1C,CHOL,TG,HDL,LDL,VLDL,BMI,CLASE
0,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
1,1,26,4.5,62,4.9,3.7,1.4,1.1,2.1,0.6,23.0,N
2,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
3,0,50,4.7,46,4.9,4.2,0.9,2.4,1.4,0.5,24.0,N
4,1,33,7.1,46,4.9,4.9,1.0,0.8,2.0,0.4,21.0,N


In [33]:
diabetes['CLASE'] = diabetes['CLASE'].str.strip()
print(diabetes['CLASE'].unique())

['N' 'P' 'Y']


In [34]:
diabetes.shape

(1000, 12)

In [35]:
X = diabetes.drop('CLASE', axis=1)
y = diabetes['CLASE']

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [37]:
X_train.shape, y_train.shape

((700, 11), (700,))

In [38]:
X_test.shape, y_test.shape

((300, 11), (300,))

Modeling

KNN

In [39]:
knn = KNeighborsClassifier(n_neighbors=49)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print("KNN:")
print(classification_report(y_test, y_pred_knn, zero_division=0))

KNN:
              precision    recall  f1-score   support

           N       0.70      0.44      0.54        36
           P       0.00      0.00      0.00        10
           Y       0.91      0.99      0.95       254

    accuracy                           0.89       300
   macro avg       0.53      0.48      0.50       300
weighted avg       0.85      0.89      0.87       300



Naive Bayes

In [40]:
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("Naive Bayes:")
print(classification_report(y_test, y_pred_nb, zero_division=0))

Naive Bayes:
              precision    recall  f1-score   support

           N       0.74      0.89      0.81        36
           P       0.82      0.90      0.86        10
           Y       0.98      0.95      0.96       254

    accuracy                           0.94       300
   macro avg       0.85      0.91      0.88       300
weighted avg       0.95      0.94      0.94       300



Support Vector Machine (SVM)

In [41]:
svm = SVC(kernel='linear', C=1.0)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print("Support Vector Machine:")
print(classification_report(y_test, y_pred_svm, zero_division=0))

Support Vector Machine:
              precision    recall  f1-score   support

           N       0.78      0.89      0.83        36
           P       0.60      0.30      0.40        10
           Y       0.96      0.96      0.96       254

    accuracy                           0.93       300
   macro avg       0.78      0.72      0.73       300
weighted avg       0.93      0.93      0.93       300



Decision Tree

In [42]:
dt = DecisionTreeClassifier(max_depth=2)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree:")
print(classification_report(y_test, y_pred_dt, zero_division=0))

Decision Tree:
              precision    recall  f1-score   support

           N       0.73      1.00      0.85        36
           P       0.43      1.00      0.61        10
           Y       1.00      0.90      0.95       254

    accuracy                           0.91       300
   macro avg       0.72      0.97      0.80       300
weighted avg       0.95      0.91      0.92       300



Random Forest

In [43]:
rf = RandomForestClassifier(n_estimators=100)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

Random Forest:
              precision    recall  f1-score   support

           N       0.97      0.92      0.94        36
           P       1.00      0.90      0.95        10
           Y       0.98      1.00      0.99       254

    accuracy                           0.98       300
   macro avg       0.99      0.94      0.96       300
weighted avg       0.98      0.98      0.98       300



Logistic Regression-in

In [44]:
log_reg = LogisticRegression(solver='saga')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)

print("Logistic Regression:")
print(classification_report(y_test, y_pred_log_reg, zero_division=0))

Logistic Regression:
              precision    recall  f1-score   support

           N       0.00      0.00      0.00        36
           P       0.00      0.00      0.00        10
           Y       0.85      1.00      0.92       254

    accuracy                           0.85       300
   macro avg       0.28      0.33      0.31       300
weighted avg       0.72      0.85      0.78       300



/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Top 3 Worst Results:

KNN x Decision Tree x Logistic Regression-in - Adjusting Imbalanced Classes

In [52]:
print("Distribuição antes do SMOTE:", y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Distribuição após do SMOTE:", y_train_smote.value_counts())

Distribuição antes do SMOTE: CLASE
Y    590
N     67
P     43
Name: count, dtype: int64
Distribuição após do SMOTE: CLASE
Y    590
N    590
P    590
Name: count, dtype: int64


KNN with SMOTE

In [55]:
knn.fit(X_train_smote, y_train_smote)
y_pred_knn = knn.predict(X_test)

print("KNN:")
print(classification_report(y_test, y_pred_knn, zero_division=0))

KNN:
              precision    recall  f1-score   support

           N       0.39      0.67      0.49        36
           P       0.17      0.50      0.25        10
           Y       1.00      0.82      0.90       254

    accuracy                           0.79       300
   macro avg       0.52      0.66      0.55       300
weighted avg       0.90      0.79      0.83       300



Decision Tree with SMOTE

In [53]:
dt.fit(X_train_smote, y_train_smote)
y_pred_dt = dt.predict(X_test)

print("Decision Tree:")
print(classification_report(y_test, y_pred_dt, zero_division=0))

Decision Tree:
              precision    recall  f1-score   support

           N       0.86      1.00      0.92        36
           P       0.43      1.00      0.61        10
           Y       1.00      0.93      0.96       254

    accuracy                           0.94       300
   macro avg       0.76      0.98      0.83       300
weighted avg       0.96      0.94      0.94       300



Logistic Regression-in with SMOTE

In [56]:
log_reg.fit(X_train_smote, y_train_smote)
y_pred_log_reg = log_reg.predict(X_test)

print("Logistic Regression:")
print(classification_report(y_test, y_pred_log_reg, zero_division=0))

Logistic Regression:
              precision    recall  f1-score   support

           N       0.38      0.64      0.47        36
           P       0.09      0.30      0.14        10
           Y       0.97      0.79      0.87       254

    accuracy                           0.75       300
   macro avg       0.48      0.58      0.49       300
weighted avg       0.87      0.75      0.80       300



/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Grid Search x Random Search - Random Forest 

Grid Search

In [61]:
param_grid = {
    "n_estimators": [50, 75, 100],
    "max_depth": [None, 5, 10, 20],
    "class_weight": [None, "balanced"]
}

grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="accuracy",     
    cv=5,
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

print("Melhores hiperparâmetros:", grid_search_rf.best_params_)
print("Cross Validation:", grid_search_rf.best_score_)

best_rf_grid = grid_search_rf.best_estimator_
y_pred_rf_grid = best_rf_grid.predict(X_test)

print("\nRandom Forest (GridSearch)")
print(classification_report(y_test, y_pred_rf_grid, zero_division=0))

Melhores hiperparâmetros: {'class_weight': None, 'max_depth': None, 'n_estimators': 50}
Cross Validation: 0.9800000000000001

Random Forest (GridSearch)
              precision    recall  f1-score   support

           N       0.97      0.97      0.97        36
           P       1.00      0.90      0.95        10
           Y       0.99      1.00      0.99       254

    accuracy                           0.99       300
   macro avg       0.99      0.96      0.97       300
weighted avg       0.99      0.99      0.99       300



Random Search

In [62]:
rf2 = RandomForestClassifier(random_state=42)

param_dist = {
    "n_estimators": np.arange(50, 75, 100),  
    "max_depth": [None, 5, 10, 20, 30],
    "class_weight": [None, "balanced"]
}

rand_search_rf = RandomizedSearchCV(
    estimator=rf2,
    param_distributions=param_dist,
    n_iter=20,            
    scoring="accuracy",
    cv=5,
    random_state=42,
    n_jobs=-1
)

rand_search_rf.fit(X_train, y_train)

print("Melhores hiperparâmetros:", rand_search_rf.best_params_)
print("Cross Validation:", rand_search_rf.best_score_)

best_rf_rand = rand_search_rf.best_estimator_
y_pred_rf_rand = best_rf_rand.predict(X_test)

print("\nRandom Forest (RandomizedSearch) - Teste:")
print(classification_report(y_test, y_pred_rf_rand, zero_division=0))


/opt/conda/lib/python3.11/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 10 is smaller than n_iter=20. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Melhores hiperparâmetros: {'n_estimators': np.int64(50), 'max_depth': None, 'class_weight': None}
Cross Validation: 0.9842857142857143

Random Forest (RandomizedSearch) - Teste:
              precision    recall  f1-score   support

           N       0.97      0.94      0.96        36
           P       1.00      0.90      0.95        10
           Y       0.99      1.00      0.99       254

    accuracy                           0.99       300
   macro avg       0.99      0.95      0.97       300
weighted avg       0.99      0.99      0.99       300

